In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# !pip install wikipedia

In [ ]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import TokenTextSplitter

# raw_documents = WikipediaLoader (query="large language model").load()
loader = WikipediaLoader(
    query="large language model",
    lang="en",
    load_max_docs=2
)

raw_documents = loader.load()
text_splitter = TokenTextSplitter (chunk_size=100, chunk_overlap=20)
documents = text_splitter.split_documents(raw_documents[:3])

print(documents[0])

In [ ]:
# import os
# from dotenv import load_dotenv

# load_dotenv()

# hf_token = os.getenv("HF_TOKEN")
# print(hf_token)  # debug once, then remove

In [ ]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


loader = WikipediaLoader(
    query="large language model",
    lang="en",
    load_max_docs=2
)

raw_documents = loader.load()
# raw_documents = WikipediaLoader(query="large language model").load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
documents = text_splitter.split_documents(raw_documents[:3])

print(documents[0].page_content)
print(documents[0].metadata)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm = ChatOpenAI(api_key="...", temperature=0, model_name="gpt-40-mini")
llm_transformer = LLMGraphTransformer(llm=llm)
graph_documents = llm_transformer.convert_to_graph_documents (documents)
print(graph_documents)

In [ ]:
# from langchain_community.llms import HuggingFaceEndpoint
from langchain_community.llms import HuggingFaceHub
from langchain_experimental.graph_transformers import LLMGraphTransformer
# llm = HuggingFaceEndpoint(
#     repo_id="mistralai/Mistral-7B-Instruct-v0.2",
#     temperature=0.1,
#     huggingfacehub_api_token=hf_token
# )

# llm = HuggingFaceEndpoint(
#     huggingfacehub_api_token=hf_token,
#     model="mistralai/Mistral-7B-Instruct-v0.2"
# )

# llm = HuggingFaceEndpoint(
#     huggingfacehub_api_token=hf_token,
#     model="mistralai/Mistral-7B-Instruct-v0.2",
#     task="text-generation",
#     model_kwargs={
#         "temperature": 0.1,
#         "max_new_tokens": 256
#     }
# )

# llm = HuggingFaceHub(
#     repo_id="mistralai/Mistral-7B-Instruct-v0.2",
#     huggingfacehub_api_token=hf_token,
#     model_kwargs={
#         "temperature": 0.1,
#         "max_new_tokens": 256
#     }
# )

llm = HuggingFaceHub(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    huggingfacehub_api_token=hf_token,
    model_kwargs={"temperature": 0.1}
)
# Graph transformer
graph_transformer = LLMGraphTransformer(llm=llm)
graph_documents = graph_transformer.convert_to_graph_documents(documents)

print(graph_documents[0])

In [ ]:
# !pip install -U huggingface_hub
# !pip install -U langchain langchain-community langchain-experimental

In [ ]:
from langchain_community.llms import HuggingFaceHub
from huggingface_hub import InferenceClient
# llm = HuggingFaceHub(
#     repo_id="mistralai/Mistral-7B-Instruct-v0.2",
#     huggingfacehub_api_token=hf_token,
#     model_kwargs={
#         "temperature": 0.1,
#         "max_new_tokens": 512
#     }
# )

client = InferenceClient(token=hf_token)

# def extract_graph(text):
#     prompt = f"""
# You are a knowledge graph extractor.

# Extract ONLY factual relationships from the text.

# Return STRICT JSON array:
# [
#   {{"head": "entity1", "relation": "relation_type", "tail": "entity2"}}
# ]

# Rules:
# - No explanation
# - No markdown
# - Only valid JSON
# - Keep relations short verbs (is, part_of, created_by, used_for)

# TEXT:
# {text}
# """
#     return llm.invoke(prompt)


# def extract_graph(text):
#     prompt = f"""
# Extract knowledge graph triples.

# Return ONLY JSON:
# [
#   {{"head": "...", "relation": "...", "tail": "..."}}
# ]

# TEXT:
# {text}
# """

#     response = client.text_generation(
#         prompt,
#         max_new_tokens=512,
#         temperature=0.1
#     )

#     return response

# def extract_graph(text):
#     messages = [
#         {
#             "role": "user",
#             "content": f"""
# Extract knowledge graph triples from the text.

# Return STRICT JSON only:
# [
#   {{"head": "entity", "relation": "relation", "tail": "entity"}}
# ]

# TEXT:
# {text}
# """
#         }
#     ]

#     response = client.chat_completion(
#         model="mistralai/Mistral-7B-Instruct-v0.2",
#         messages=messages,
#         temperature=0.1,
#         max_tokens=512
#     )

#     return response.choices[0].message.content

def extract_graph(text):
    messages = [
        {
            "role": "user",
            "content": f"""
Extract knowledge graph triples.

Return STRICT JSON:
[
  {{"head": "...", "relation": "...", "tail": "..."}}
]

TEXT:
{text}
"""
        }
    ]

    response = client.chat_completion(
        model="HuggingFaceH4/zephyr-7b-beta",
        messages=messages,
        max_tokens=512,
        temperature=0.1
    )

    return response.choices[0].message.content

import json
import re

# def safe_parse(response):
#     try:
#         return json.loads(response)
#     except:
#         return []
    

def safe_parse(response):
    try:
        # extract JSON block from messy LLM output
        match = re.search(r"\[.*\]", response, re.DOTALL)
        if match:
            return json.loads(match.group())
        return []
    except:
        return []

graph_data = []

for doc in documents:
    response = extract_graph(doc.page_content)
    triples = safe_parse(response)
    graph_data.extend(triples)

print(graph_data[:5])

import networkx as nx

G = nx.DiGraph()

for t in graph_data:
    G.add_edge(t["head"], t["tail"], relation=t["relation"])

In [ ]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from huggingface_hub import InferenceClient
import networkx as nx
import json
import re

# =========================
# 1. LOAD DATA
# =========================
loader = WikipediaLoader(
    query="large language model",
    lang="en",
    load_max_docs=2
)

raw_documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

documents = splitter.split_documents(raw_documents)

# =========================
# 2. HF CLIENT
# =========================
client = InferenceClient(token=hf_token)

MODEL = "HuggingFaceH4/zephyr-7b-beta"

# =========================
# 3. SAFE GRAPH EXTRACTION
# =========================
def extract_graph(text):
    try:
        messages = [
            {
                "role": "user",
                "content": f"""
Extract knowledge graph triples.

STRICT RULES:
- ONLY JSON array
- NO explanation
- NO markdown
- NO text before/after

FORMAT:
[
  {{"head": "entity", "relation": "relation", "tail": "entity"}}
]

TEXT:
{text}
"""
            }
        ]

        response = client.chat_completion(
            model="HuggingFaceH4/zephyr-7b-beta",
            messages=messages,
            temperature=0.0,
            max_tokens=512
        )

        if not response or not response.choices:
            return ""

        return response.choices[0].message.content or ""

    except Exception:
        return ""
# =========================
# 4. ROBUST JSON PARSER
# =========================

def safe_parse(response):
    if not response:
        return []

    try:
        # Step 1: extract JSON block safely
        start = response.find("[")
        end = response.rfind("]")

        if start == -1 or end == -1:
            return []

        json_str = response[start:end+1]

        # Step 2: clean artifacts
        json_str = json_str.replace("```json", "").replace("```", "").strip()

        return json.loads(json_str)

    except Exception:
        return []
# =========================
# 5. BUILD GRAPH DATA
# =========================
graph_data = []
failed = 0

for doc in documents:
    response = extract_graph(doc.page_content)

    if not response:
        failed += 1
        continue

    triples = safe_parse(response)

    if not triples:
        failed += 1

    graph_data.extend(triples)

print("Failed chunks:", failed)


# =========================
# 6. BUILD GRAPH
# =========================
def normalize(x):
    return x.lower().strip()

G = nx.DiGraph()

for t in graph_data:
    try:
        head = normalize(t.get("head", ""))
        tail = normalize(t.get("tail", ""))
        rel = t.get("relation", "").lower().strip()

        if head and tail:
            G.add_edge(head, tail, relation=rel)

    except:
        continue

for doc in documents:
    response = extract_graph(doc.page_content)

    print("\nRAW RESPONSE:\n", response[:300])  # IMPORTANT DEBUG

    triples = safe_parse(response)
    graph_data.extend(triples)
# =========================
# 7. OUTPUT STATS
# =========================
print("\n===== GRAPH STATS =====")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

print("\nSample edges:")
print(list(G.edges(data=True))[:10])

In [ ]:
# !pip install neo4j

In [ ]:
import os
from langchain_community.graphs import Neo4jGraph
from dotenv import load_dotenv


load_dotenv()
url = os.environ ["NEO4J_URI"]
user = os.environ ["NEO4J_USERNAME"]
password = os.environ ["NEO4J_PASSWORD"]
graph = Neo4jGraph(url=url, username=user, password=password, database="f0159d9a")
graph.query("RETURN 1")

# from neo4j import GraphDatabase

# # URI examples: "neo4j://localhost", "neo4j+s://xxx.databases.neo4j.io"
# URI = "neo4j+s://f0159d9a.databases.neo4j.io"
# AUTH = (user,password)

# with GraphDatabase.driver(URI, auth=AUTH) as driver:
#     driver.verify_connectivity()

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm = ChatOpenAI(api_key="...", temperature=0, model="gpt-40-mini")
llm_transformer = LLMGraphTransformer(llm=llm)
graph_documents = llm_transformer.convert_to_graph_documents (documents)

In [ ]:
from huggingface_hub import InferenceClient
import json
import re

client = InferenceClient(token="YOUR_HF_TOKEN")

MODEL = "HuggingFaceH4/zephyr-7b-beta"


# =========================
# 1. HF GRAPH EXTRACTOR
# =========================
def extract_graph(text):
    prompt = f"""
You are a knowledge graph extraction system.

Return ONLY valid JSON array:

[
  {{"head": "entity", "relation": "relation", "tail": "entity"}}
]

Rules:
- no explanation
- no markdown
- only JSON
- relation must be short verb (is, part_of, used_for)

TEXT:
{text}
"""

    response = client.chat_completion(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=512
    )

    return response.choices[0].message.content


# =========================
# 2. SAFE PARSER
# =========================
def safe_parse(response):
    if not response:
        return []

    try:
        start = response.find("[")
        end = response.rfind("]")

        if start == -1 or end == -1:
            return []

        json_str = response[start:end+1]
        json_str = json_str.replace("```json", "").replace("```", "")

        return json.loads(json_str)

    except:
        return []


# =========================
# 3. REPLACEMENT FOR convert_to_graph_documents
# =========================
def hf_convert_to_graph_documents(documents):
    graph_data = []

    for doc in documents:
        response = extract_graph(doc.page_content)
        triples = safe_parse(response)

        for t in triples:
            graph_data.append({
                "source": t.get("head"),
                "target": t.get("tail"),
                "relation": t.get("relation")
            })

    return graph_data

In [ ]:
graph_documents = hf_convert_to_graph_documents(documents)

print(graph_documents[:5])

In [ ]:
graph.add_graph_documents(
graph_documents,
include_source=True,
baseEntityLabel=True
)

In [ ]:
print(graph.get_schema)


In [ ]:
results = graph.query("""
MATCH (gpt4: Model {id: "Gpt-4"})-[:DEVELOPED_BY]-> :Organization)
RETURN org""")
print(results)

In [ ]:
# Instantiate the Neo4j graph
graph = Neo4jGraph(url=url, username=user, password=password)

# Add the graph documents, sources, and include entity labels
graph.add_graph_documents(
  graph_documents, 
  include_source=True, 
  baseEntityLabel=True
)
graph.refresh_schema()

# Print the graph schema
print(graph.get_schema)

In [ ]:
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain
chain = GraphCypherQAChain.from_llm(
llm=ChatOpenAI (api_key="...", temperature=0), graph=graph, verbose=True
)
result = chain.invoke({"query": "What is the most accurate model?"})

In [ ]:
print(f"Final answer: {result['result']}")

In [ ]:
chain = GraphCypherQAChain.from_llm(
llm=ChatOpenAI(api_key="...", temperature=0), graph=graph, verbose=True
)

In [ ]:
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain

llm = ChatOpenAI(api_key="...", model="gpt-40-mini", temperature=0)

chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, exclude_types=["Concept"], verbose=True)
chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, exclude_types=["Concept"], verbose=True,validate_cypher=True)
print(graph.get_schema)

In [ ]:
examples = [
    {
        "question": "How many notable large language models are mentioned in the article?",
        "query": "MATCH (m:Concept {id: 'Large Language Model'}) RETURN count(DISTINCT m)",
    },
    {
        "question": "Which companies or organizations have developed the large language models mentioned?",
        "query": "MATCH (o:Organization)-[:DEVELOPS]->(m:Concept {id: 'Large Language Model'}) RETURN DISTINCT o.id",
    },
    {
        "question": "What is the largest model size mentioned in the article, in terms of number of parameters?",
        "query": "MATCH (m:Concept {id: 'Large Language Model'}) RETURN max(m.parameters) AS largest_model",
    },
]


In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
example_prompt = PromptTemplate.from_template(
"User input: {question}\nCypher query: {query}"
)
cypher_prompt = FewShotPromptTemplate(
examples=examples,
example_prompt=example_prompt,
prefix="You are a Neo4j expert. Given an input question, create a syntactically correct Cypher query to run.\n\nHere is the schema information\n{schema}.\n\nBelow are a number of examples of questions and their corresponding Cypher queries.",
suffix="User input: {question}\nCypher query: ",
input_variables=["question"],
)

In [ ]:
chain = GraphCypherQAChain.from_llm(
graph=graph, llm=llm, cypher_prompt=cypher_prompt,
verbose=True, validate_cypher=True
)